# QLoRA fine-tuning ? Colab T4

The adapter is trained on synthetic question/answer pairs without retrieved context. Training checkpoints and outputs are saved to Google Drive.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Select a GPU runtime in Runtime > Change runtime type"

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, json, shutil, time

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/farmer-llm')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
REPO_URL = 'https://github.com/sarthakmehra02/stuff.git'
REPO = Path('/content/farmer-llm-project')
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)


In [ ]:
%pip install -r requirements-colab.txt
import importlib.metadata as m
for name in ['torch','transformers','peft','trl','bitsandbytes','datasets']:
    try: print(f'{name}={m.version(name)}')
    except m.PackageNotFoundError: print(f'{name}=not installed')


In [ ]:
%%time
!python src/finetune.py --train data/train_split/train.jsonl --val data/train_split/val.jsonl --out /content/drive/MyDrive/farmer-llm/run1 --smoke


## Smoke run check

Confirm the 20-step smoke run finishes and the masked prompt/answer preview looks correct before running full training.

In [ ]:
%%time
!python src/finetune.py --train data/train_split/train.jsonl --val data/train_split/val.jsonl --out /content/drive/MyDrive/farmer-llm/run1 --epochs 3
from pathlib import Path
print(Path('/content/drive/MyDrive/farmer-llm/run1/training_summary.json').read_text())
